# Training Progress & Model Analysis (Colab-Ready)

This notebook orchestrates lightweight self-play training runs, automatic checkpointing, and post-run diagnostics for the Texas Hold'em AI project. It is designed to run in Google Colab or a local Jupyter environment so you can verify that training is making forward progress while capturing the artefacts needed for deeper analysis.


## 1. Configure the execution environment

Run the next cell to ensure the repository is available. When using Google Colab, set the `POKER_AI_REPO_URL` environment variable to point to your fork (for example by running `import os; os.environ['POKER_AI_REPO_URL'] = 'https://github.com/my-user/texas_holdem_ai_gatc.git'`). If the repository is already checked out locally, the cell simply reuses the existing working tree.


In [ ]:

import os
import subprocess
import sys
from pathlib import Path

DEFAULT_REPO_URL = "https://github.com/YOUR_ACCOUNT/texas_holdem_ai_gatc.git"
REPO_NAME = "texas_holdem_ai_gatc"

repo_root = Path.cwd()
if not (repo_root / "src" / "poker_ai").exists():
    target_dir = repo_root / REPO_NAME
    if not target_dir.exists():
        repo_url = os.environ.get("POKER_AI_REPO_URL", DEFAULT_REPO_URL)
        if "YOUR_ACCOUNT" in repo_url:
            raise RuntimeError(
                "Set the 'POKER_AI_REPO_URL' environment variable to the Git URL of your fork before running this cell (for example: "https://github.com/my-user/texas_holdem_ai_gatc.git")."
            )
        print(f"Cloning repository from {repo_url} ...")
        subprocess.run(["git", "clone", repo_url, str(target_dir)], check=True)
    repo_root = target_dir
    os.chdir(repo_root)
else:
    os.chdir(repo_root)
    repo_root = Path.cwd()

os.environ["POKER_AI_REPO_ROOT"] = str(repo_root)
print(f"Repository root: {repo_root}")
sys.path.insert(0, str(repo_root / "src"))


In [ ]:

#@title Install dependencies
import subprocess
import sys
from pathlib import Path

REQUIREMENTS_PATH = Path("requirements.txt")
commands: list[list[str]] = []
if REQUIREMENTS_PATH.exists():
    commands.append([sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_PATH)])
extra_packages = ["pandas", "matplotlib", "seaborn"]
commands.append([sys.executable, "-m", "pip", "install", "--quiet", *extra_packages])

for cmd in commands:
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)


In [ ]:

#@title Prepare run directories and a lightweight config
from __future__ import annotations

import json
import os
from copy import deepcopy
from pathlib import Path

import yaml

REPO_ROOT = Path(os.environ.get("POKER_AI_REPO_ROOT", Path.cwd()))
RUN_NAME = os.environ.get("TRAINING_RUN_NAME", "colab_progress_check")
OUTPUT_DIR = REPO_ROOT / "colab_runs" / RUN_NAME
MODELS_DIR = OUTPUT_DIR / "models"
LOG_DIR = OUTPUT_DIR / "logs"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

base_config_path = REPO_ROOT / "src" / "poker_ai" / "config" / "config.yaml"
with base_config_path.open() as fh:
    base_config = yaml.safe_load(fh)

config = deepcopy(base_config)
training_cfg = config.setdefault("training", {})
training_cfg["num_training_hands"] = min(int(training_cfg.get("num_training_hands", 1000)), 1000)
training_cfg["save_model_path"] = str(MODELS_DIR / "latest.pth")
training_cfg["save_model_every_n_hands"] = 0
training_cfg["save_model_every_minutes"] = 0
training_cfg["save_model_every_samples"] = 128
training_cfg["min_buffer_before_train"] = max(32, int(training_cfg.get("min_buffer_before_train", 64)))
training_cfg["samples_per_cycle"] = 128
training_cfg["train_steps_per_cycle"] = 1
training_cfg["train_batch_size"] = 128

logging_cfg = config.setdefault("logging", {})
logging_cfg["log_to_file"] = True
logging_cfg["log_file"] = str(LOG_DIR / "training.log")
logging_cfg["log_dir"] = str(LOG_DIR)
logging_cfg["level"] = logging_cfg.get("level", "INFO")
logging_cfg.setdefault("format", "%(asctime)s | %(levelname)s | %(component)s | %(run_id)s | %(name)s | %(message)s")

config_path = OUTPUT_DIR / "training_config.yaml"
with config_path.open("w") as fh:
    yaml.safe_dump(config, fh)

analysis_state = {
    "config_path": str(config_path),
    "models_dir": str(MODELS_DIR),
    "log_dir": str(LOG_DIR),
    "samples_per_cycle": training_cfg["samples_per_cycle"],
    "save_every_samples": training_cfg["save_model_every_samples"],
    "train_batch_size": training_cfg["train_batch_size"],
}
analysis_state_path = OUTPUT_DIR / "analysis_state.json"
with analysis_state_path.open("w") as fh:
    json.dump(analysis_state, fh, indent=2)

os.environ["TRAINING_ANALYSIS_STATE"] = str(analysis_state_path.resolve())

print(json.dumps(analysis_state, indent=2))


## 2. Run a lightweight training session

Toggle `RUN_TRAINING` to `True` when you are ready to launch a short training loop. The defaults keep the run CPU-friendly by limiting the number of self-play samples and forcing the model to save snapshots every few dozen iterations.


In [ ]:

       #@title Launch training (enable RUN_TRAINING to execute)
       import json
       import os
       import shlex
       import subprocess
       import sys
       from pathlib import Path

       RUN_TRAINING = False  #@param {type:"boolean"}
       MAX_SAMPLES = 512  #@param {type:"integer"}

       analysis_state_path = Path(os.environ["TRAINING_ANALYSIS_STATE"])
       with analysis_state_path.open() as fh:
           analysis_state = json.load(fh)

       REPO_ROOT = Path(os.environ.get("POKER_AI_REPO_ROOT", Path.cwd()))
       config_path = Path(analysis_state["config_path"])
       models_dir = Path(analysis_state["models_dir"])
       models_dir.mkdir(parents=True, exist_ok=True)

       if RUN_TRAINING:
           cmd = [
               sys.executable,
               "-m",
               "poker_ai.cli.train",
               "--config",
               str(config_path),
               "--algorithm",
               "ai_cfr",
               "--samples-per-cycle",
               str(analysis_state["samples_per_cycle"]),
               "--train-steps-per-cycle",
               "1",
               "--train-batch-size",
               str(analysis_state["train_batch_size"]),
               "--save-samples",
               str(analysis_state["save_every_samples"]),
               "--max-samples",
               str(MAX_SAMPLES),
               "--save-model-every",
               "0",
               "--save-minutes",
               "0",
           ]
           env = os.environ.copy()
           env.setdefault("PYTHONUNBUFFERED", "1")
           print("Executing:
", " ".join(shlex.quote(part) for part in cmd))
           subprocess.run(cmd, check=True, env=env, cwd=REPO_ROOT)
       else:
           print("Training skipped. Set RUN_TRAINING = True above when you are ready to run a session.")


## 3. Parse training logs and checkpoint metadata

After a run completes, the following cells reshape the structured log output and saved checkpoint files into tables and plots. This helps confirm that losses trend downward and that new models are being saved when improvements are detected.


In [ ]:

#@title Convert log output into tabular data
import json
import os
import re
from datetime import datetime
from pathlib import Path

import pandas as pd

analysis_state_path = Path(os.environ["TRAINING_ANALYSIS_STATE"])
with analysis_state_path.open() as fh:
    analysis_state = json.load(fh)

log_dir = Path(analysis_state["log_dir"])
log_files = sorted(log_dir.glob("*.log"))
if not log_files:
    raise FileNotFoundError(f"No log files found in {log_dir}. Run a training session first.")
log_file = log_files[-1]
print(f"Reading logs from {log_file}")

pattern_cycle = re.compile(r"Starting cycle (?P<cycle>\d+) \| generating (?P<samples>\d+) samples \(total so far (?P<total>\d+)\)")
pattern_loss = re.compile(r"Cycle (?P<cycle>\d+) \| completed (?P<steps>\d+) training steps \| avg loss=(?P<loss>[-+\d.eE]+)")
pattern_steps = re.compile(r"Cycle (?P<cycle>\d+) \| completed (?P<steps>\d+) training steps$")

cycle_summary: dict[int, dict[str, int]] = {}
loss_rows = []
step_rows = []
save_rows = []
analyzer_rows = []

pattern_save = re.compile(r"Model saved to (?P<path>\S+) (?:at sample|due to time interval at sample|at iteration) (?P<count>\d+)")
pattern_analyzer_save = re.compile(r"Model saved to (?P<path>\S+) at iteration (?P<count>\d+)")
pattern_pool = re.compile(r"Model pool size (?P<size>\d+)/(?:\d+); retaining new snapshot (?P<path>\S+)")
pattern_pool_drop = re.compile(r"Removed model (?P<path>\S+) from pool")
pattern_warning = re.compile(r"Stopping training early after (?P<count>\d+) samples with no improvement\.")

with log_file.open() as fh:
    for line in fh:
        try:
            timestamp_str, _, _, _, _, message = line.strip().split(" | ", 5)
        except ValueError:
            message = line.strip()
            timestamp_str = None
        timestamp = None
        if timestamp_str:
            try:
                timestamp = datetime.fromisoformat(timestamp_str)
            except ValueError:
                pass

        match_cycle = pattern_cycle.search(message)
        if match_cycle:
            cycle = int(match_cycle.group("cycle"))
            start_total = int(match_cycle.group("total"))
            generated = int(match_cycle.group("samples"))
            cycle_summary[cycle] = {
                "start_total": start_total,
                "generated": generated,
                "end_total": start_total + generated,
            }
            continue

        if (match := pattern_loss.search(message)):
            cycle = int(match.group("cycle"))
            loss_rows.append(
                {
                    "timestamp": timestamp,
                    "cycle": cycle,
                    "steps": int(match.group("steps")),
                    "avg_loss": float(match.group("loss")),
                    "samples_seen": cycle_summary.get(cycle, {}).get("end_total"),
                }
            )
            continue

        if (match := pattern_steps.search(message)):
            cycle = int(match.group("cycle"))
            step_rows.append(
                {
                    "timestamp": timestamp,
                    "cycle": cycle,
                    "steps": int(match.group("steps")),
                    "samples_seen": cycle_summary.get(cycle, {}).get("end_total"),
                }
            )
            continue

        if (match := pattern_analyzer_save.search(message)):
            save_rows.append(
                {
                    "timestamp": timestamp,
                    "samples_seen": int(match.group("count")),
                    "path": match.group("path"),
                    "source": "analyzer",
                }
            )
            continue

        if (match := pattern_save.search(message)):
            save_rows.append(
                {
                    "timestamp": timestamp,
                    "samples_seen": int(match.group("count")),
                    "path": match.group("path"),
                    "source": "manual",
                }
            )
            continue

        if (match := pattern_pool.search(message)):
            analyzer_rows.append(
                {
                    "timestamp": timestamp,
                    "event": "retain",
                    "path": match.group("path"),
                    "pool_size": int(match.group("size")),
                }
            )
            continue

        if (match := pattern_pool_drop.search(message)):
            analyzer_rows.append(
                {
                    "timestamp": timestamp,
                    "event": "drop",
                    "path": match.group("path"),
                    "pool_size": None,
                }
            )
            continue

        if (match := pattern_warning.search(message)):
            analyzer_rows.append(
                {
                    "timestamp": timestamp,
                    "event": "early_stop",
                    "samples_seen": int(match.group("count")),
                }
            )
            continue

loss_df = pd.DataFrame(loss_rows)
step_df = pd.DataFrame(step_rows)
saves_df = pd.DataFrame(save_rows)
analyzer_events_df = pd.DataFrame(analyzer_rows)

display({"losses": loss_df, "steps": step_df, "saves": saves_df, "analyzer_events": analyzer_events_df})


In [ ]:

#@title Visualise progress metrics
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

if 'loss_df' not in globals():
    raise RuntimeError("Run the log parsing cell first.")

sns.set_theme(style="whitegrid")

if loss_df.empty:
    print("No averaged loss records were found. Increase logging verbosity or rerun training.")
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    plot_df = loss_df.dropna(subset=["samples_seen"]).sort_values("samples_seen")
    sns.lineplot(data=plot_df, x="samples_seen", y="avg_loss", marker="o", ax=ax)
    ax.set_xlabel("Self-play samples processed")
    ax.set_ylabel("Average loss per cycle")
    ax.set_title("Training loss across cycles")
    fig.tight_layout()
    display(fig)

if 'saves_df' in globals() and not saves_df.empty:
    fig2, ax2 = plt.subplots(figsize=(8, 2.75))
    saves_plot = saves_df.dropna(subset=["samples_seen"]).copy()
    saves_plot["source_label"] = saves_plot["source"].map({"analyzer": "Analyzer checkpoint", "manual": "Interval checkpoint"})
    sns.scatterplot(data=saves_plot, x="samples_seen", y="source_label", hue="source_label", style="source_label", s=120, ax=ax2)
    ax2.set_xlabel("Self-play samples processed")
    ax2.set_ylabel("")
    ax2.set_title("Checkpoint timeline")
    ax2.legend(loc="upper center", bbox_to_anchor=(0.5, 1.35), ncol=2)
    fig2.tight_layout()
    display(fig2)
else:
    print("No checkpoint save events were parsed.")


In [ ]:

#@title Inspect saved models and (optionally) run a mini-tournament
import json
import os
from pathlib import Path

import pandas as pd
import torch

analysis_state_path = Path(os.environ["TRAINING_ANALYSIS_STATE"])
with analysis_state_path.open() as fh:
    analysis_state = json.load(fh)

models_dir = Path(analysis_state["models_dir"])
model_paths = sorted(models_dir.glob("*.pth"))
records = []
for path in model_paths:
    stats = path.stat()
    payload_info: dict[str, object] = {}
    try:
        payload = torch.load(path, map_location="cpu")
        if isinstance(payload, dict):
            payload_info["keys"] = sorted(payload.keys())
            metadata = payload.get("metadata")
            if isinstance(metadata, dict):
                payload_info["metadata"] = {
                    k: metadata[k]
                    for k in ("hidden_dim", "num_layers", "num_heads", "normalization_scale")
                    if k in metadata
                }
        else:
            payload_info["type"] = type(payload).__name__
    except Exception as exc:
        payload_info["error"] = str(exc)
    records.append(
        {
            "path": str(path),
            "size_mb": round(stats.st_size / (1024 * 1024), 3),
            "modified": stats.st_mtime,
            "details": payload_info,
        }
    )

models_df = pd.DataFrame(records)
display(models_df)

RUN_TOURNAMENT = False  #@param {type:"boolean"}
GAMES_PER_MATCH = 2  #@param {type:"integer"}

if RUN_TOURNAMENT:
    if len(model_paths) < 2:
        print("Need at least two checkpoints to run a tournament.")
    else:
        from poker_ai.evaluation.performance_analysis import run_tournament

        scores = run_tournament([str(path) for path in model_paths], games_per_match=GAMES_PER_MATCH)
        display(scores)
else:
    print("Mini-tournament skipped. Enable RUN_TOURNAMENT to evaluate saved models head-to-head.")


## 4. Next steps

* Adjust the sampling and batch-size parameters in the configuration cell to explore longer or more aggressive runs.
* Enable `RUN_TOURNAMENT` to keep only the strongest checkpoints and quickly spot regressions.
* Upload the `colab_runs/<run_name>` directory to persistent storage (e.g., Google Drive) to archive logs, configs, and models for future comparisons.
